# 00 · Build the evaluation splits

Runs on **CPU** — no GPU needed, so use a CPU runtime and save your GPU quota.

Produces:
* `split_a.jsonl` — 125 geometrically-deduplicated DeepCAD test shapes × L0–L3 = **500 prompts**
* `split_b.jsonl` — 200 CADPrompt objects × 2 prompt variants = **400 prompts**, each flagged
  `clean` or `contaminated`

The dedup pass is the slow part (~30 min for 8k meshes on 16 processes) and is cached,
so re-running is cheap.

In [ ]:
# Mount Drive so predictions survive a Colab timeout, then get the harness.
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/t2c_bench'
os.makedirs(WORK, exist_ok=True)
os.environ['T2C_WORK'] = WORK

!git clone -q https://github.com/prashantkul366/T2C_Benchamrk /content/t2cbench_repo || (cd /content/t2cbench_repo && git pull -q)
%cd /content/t2cbench_repo
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
print('work dir:', WORK)

In [ ]:
# CPU-side evaluation dependencies. embreex matters: without it the exact
# point-in-solid test falls back to a pure-Python ray engine and voxel IoU goes
# from ~0.5s to ~90s per sample.
!pip install -q trimesh rtree embreex manifold3d scipy pandas matplotlib tabulate pyyaml tqdm
!pip install -q cadquery
import trimesh, embreex
print('trimesh', trimesh.__version__, '| embreex present')

### CADPrompt

Cloned from the paper's repo. Its 200 directories are named with DeepCAD uids, which is
what lets us check contamination at all.

In [ ]:
!git clone -q --depth 1 https://github.com/Kamel773/CAD_Code_Generation /content/CADPrompt || true
CADPROMPT = '/content/CAD_Code_Generation/CADPrompt'
print(len(os.listdir(CADPROMPT)), 'CADPrompt objects')

### Build both splits

The Text2CAD L0–L3 prompts come from `text2cad_v1.1.csv`. The canonical copy is in the
**gated** `SadilKhan/Text2CAD` repo; `ricemonster/NeurIPS11092` mirrors the same file
ungated, which is what the builder uses by default, so this step needs no token.

In [ ]:
!python -m t2cbench.data.build_splits \
    --stage all \
    --out $T2C_WORK/data \
    --cache $T2C_WORK/hf_cache \
    --cadprompt-dir /content/CAD_Code_Generation/CADPrompt \
    --n-uids 125 --workers 16

### Sanity-check what was built

In [ ]:
import json, collections, pandas as pd
a = [json.loads(l) for l in open(f'{WORK}/data/split_a.jsonl')]
b = [json.loads(l) for l in open(f'{WORK}/data/split_b.jsonl')]
print(f'Split A: {len(a)} prompts over {len({r["uid"] for r in a})} uids')
print('  by level     :', dict(collections.Counter(r['level'] for r in a)))
print('  by complexity:', dict(collections.Counter(r['complexity_bin'] for r in a)))
print(f'Split B: {len(b)} prompts over {len({r["uid"] for r in b})} uids')
print('  contamination:', dict(collections.Counter(r['contamination'] for r in b)))

ex = a[0]
print(f"\nExample -- {ex['sample_id']} ({ex['complexity_bin']})")
for lv in ['L0','L1','L2','L3']:
    p = next(r['prompt'] for r in a if r['uid']==ex['uid'] and r['level']==lv)
    print(f"  {lv}: {p[:150]}{'...' if len(p)>150 else ''}")

Both split files now live in Drive and every later notebook reads them from there.
**Do not rebuild them between model runs** — a different split means the models are no
longer being compared on the same thing.